# Zero-Shot Low-Light Image Enhancement

CLIP semantic guidance + Stable Diffusion (DDIM) for zero-shot low-light image enhancement. No paired training data is used; both CLIP and Stable Diffusion are pretrained and used as-is.

**Pipeline**: traditional gamma/color enhancement -> CLIP-guided diffusion refinement -> hybrid blend of the two outputs.

**Sections**:
1. Setup
2. Core enhancement pipeline
3. Run enhancement (single image or batch)
4. Evaluation (PSNR, SSIM, LPIPS, MUSIQ, LOE, FID)

## 1. Setup

In [ ]:
# Environment setup
# Installs PyTorch, CLIP, and Stable Diffusion dependencies.

!pip install -U pip setuptools wheel
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install git+https://github.com/openai/CLIP.git
!pip install diffusers transformers accelerate safetensors
!pip install opencv-python-headless pillow pyiqa


In [ ]:
# Mount Google Drive for dataset and output access.

from google.colab import drive
drive.mount('/content/drive')


## 2. Core enhancement pipeline

In [ ]:
# Zero-shot CLIP-guided diffusion low-light enhancement.
# Core configuration and enhancement pipeline.

import torch
import torch.nn.functional as F
import numpy as np
import cv2
import clip
import gc
from PIL import Image
from typing import Tuple, Dict, Optional
from diffusers import StableDiffusionImg2ImgPipeline, DDIMScheduler
import matplotlib.pyplot as plt
import warnings
import os
import datetime

warnings.filterwarnings("ignore")


class Config:
    """Default enhancement parameters. Override in the next cell for a given run."""

    # Image sizing
    MIN_SIZE = 512
    MAX_SIZE = 640

    # Traditional preprocessing (gamma correction)
    TARGET_LUMINANCE = 175
    GAMMA_MIN = 0.6
    GAMMA_MAX = 1.2

    # Diffusion parameters
    NUM_INFERENCE_STEPS = 40
    GUIDANCE_SCALE = 3.0
    DIFFUSION_STRENGTH = 0.25
    NUM_ITERATIONS = 4

    # Semantic preservation thresholds
    MIN_PRESERVATION_SCORE = 0.85
    EARLY_STOP_SCORE = 0.30

    # Color and contrast adjustment
    SATURATION_BOOST = 1.30
    CONTRAST_BOOST = 1.3
    PRESERVATION_WEIGHT = 0.75
    BRIGHTNESS_BOOST = 1.2

    # Traditional/diffusion blend
    HYBRID_BLEND_RATIO = 0.65
    DETAIL_PRESERVATION_STRENGTH = 0.85

    # Brightness-adaptive thresholds (mean V-channel value)
    VERY_DARK_THRESHOLD = 50
    DARK_THRESHOLD = 70
    MODERATE_DARK_THRESHOLD = 100

    # Diffusion guidance prompts
    TARGET_PROMPT = (
        "a crystal clear, vivid, well-lit photograph with vibrant colors, "
        "rich details and natural sharpness, bright and luminous"
    )
    NEGATIVE_PROMPT = (
        "blurry, washed out, desaturated, gray, hazy, low contrast, dark, "
        "underexposed, dim"
    )


class CLIPGuidedDiffusionEnhancer:
    """Zero-shot low-light enhancement using CLIP semantic guidance and latent diffusion.

    Pipeline:
      1. Traditional enhancement (adaptive gamma + color boost) as initialization
      2. CLIP-guided diffusion refinement over several candidate iterations
      3. Hybrid blend of the traditional and diffusion outputs
    """

    def __init__(self, config: Config):
        self.config = config
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.dtype = torch.float16 if self.device == "cuda" else torch.float32

        print("Loading CLIP...")
        self.clip_model, self.clip_preprocess = clip.load("ViT-B/32", device=self.device)
        self.clip_model.train()

        self.pos_text = self._encode_text(config.TARGET_PROMPT)
        self.neg_text = self._encode_text(config.NEGATIVE_PROMPT)

        print("Loading Stable Diffusion...")
        self.pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=self.dtype,
            safety_checker=None,
        ).to(self.device)

        self.pipe.scheduler = DDIMScheduler.from_config(self.pipe.scheduler.config)
        self.pipe.enable_attention_slicing()
        if self.device == "cuda":
            self.pipe.enable_vae_slicing()

        # Per-image dynamic parameters, set by _analyze_brightness during enhance()
        self.current_target_luminance = config.TARGET_LUMINANCE
        self.current_gamma_max = config.GAMMA_MAX
        self.current_saturation_boost = config.SATURATION_BOOST
        self.current_contrast_boost = config.CONTRAST_BOOST

        print("Model ready.\n")

    # ------------------------------------------------------------------
    # CLIP utilities
    # ------------------------------------------------------------------

    def _encode_text(self, text: str) -> torch.Tensor:
        tokens = clip.tokenize([text]).to(self.device)
        with torch.no_grad():
            emb = self.clip_model.encode_text(tokens)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb

    def _encode_image_clip(self, img: Image.Image) -> torch.Tensor:
        tensor = self.clip_preprocess(img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            emb = self.clip_model.encode_image(tensor)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb

    def _compute_clip_score(self, img: Image.Image) -> Tuple[float, float, float]:
        """Returns (clip_score, positive_similarity, negative_similarity)."""
        img_emb = self._encode_image_clip(img)
        pos_sim = (img_emb @ self.pos_text.T).item()
        neg_sim = (img_emb @ self.neg_text.T).item()
        return pos_sim - neg_sim, pos_sim, neg_sim

    # ------------------------------------------------------------------
    # Brightness-adaptive parameter selection
    # ------------------------------------------------------------------

    def _analyze_brightness(self, img: Image.Image) -> Dict[str, float]:
        """Inspects the V-channel histogram and selects enhancement strength accordingly."""
        np_img = np.array(img)
        hsv = cv2.cvtColor(np_img, cv2.COLOR_RGB2HSV)
        mean_v = hsv[:, :, 2].mean()

        hist = cv2.calcHist([hsv], [2], None, [256], [0, 256])
        percentile_10 = np.where(np.cumsum(hist) > 0.1 * np.sum(hist))[0][0]
        percentile_50 = np.where(np.cumsum(hist) > 0.5 * np.sum(hist))[0][0]

        print("Brightness analysis:")
        print(f"  Mean V-channel:   {mean_v:.1f}")
        print(f"  10th percentile:  {percentile_10:.1f}")
        print(f"  Median (50th):    {percentile_50:.1f}")

        params = {
            "target_luminance": self.config.TARGET_LUMINANCE,
            "saturation_boost": self.config.SATURATION_BOOST,
            "contrast_boost": self.config.CONTRAST_BOOST,
            "gamma_max": self.config.GAMMA_MAX,
            "brightness_level": "normal",
        }

        if mean_v < self.config.VERY_DARK_THRESHOLD:
            print(f"  Level: extremely dark (V={mean_v:.1f})")
            params.update(target_luminance=190, saturation_boost=1.25, contrast_boost=1.2,
                          gamma_max=1.2, brightness_level="extremely_dark")
        elif mean_v < self.config.DARK_THRESHOLD:
            print(f"  Level: very dark (V={mean_v:.1f})")
            params.update(target_luminance=180, saturation_boost=1.25, contrast_boost=1.2,
                          gamma_max=1.15, brightness_level="very_dark")
        elif mean_v < self.config.MODERATE_DARK_THRESHOLD:
            print(f"  Level: moderately dark (V={mean_v:.1f})")
            params.update(target_luminance=175, saturation_boost=1.25, contrast_boost=1.2,
                          gamma_max=1.12, brightness_level="moderately_dark")
        else:
            print(f"  Level: normal (V={mean_v:.1f})")
            params.update(target_luminance=170, saturation_boost=1.25, contrast_boost=1.2,
                          brightness_level="normal")

        return params

    # ------------------------------------------------------------------
    # Traditional enhancement (Stage 1)
    # ------------------------------------------------------------------

    def _adaptive_gamma(self, img: Image.Image) -> Image.Image:
        np_img = np.array(img)
        hsv = cv2.cvtColor(np_img, cv2.COLOR_RGB2HSV)
        mean_v = hsv[:, :, 2].mean()

        cur = mean_v / 255.0
        tgt = min(self.current_target_luminance / 255.0, 0.80)

        gamma = np.log(tgt) / np.log(max(cur, 1e-4))
        gamma = np.clip(gamma, self.config.GAMMA_MIN, self.current_gamma_max)

        table = np.array([(i / 255.0) ** gamma * 255 for i in range(256)]).astype("uint8")
        corrected = cv2.LUT(np_img, table)
        return Image.fromarray(corrected)

    def _boost_colors(self, img: Image.Image) -> Image.Image:
        from PIL import ImageEnhance

        img = ImageEnhance.Color(img).enhance(self.current_saturation_boost)
        img = ImageEnhance.Contrast(img).enhance(self.current_contrast_boost)
        img = ImageEnhance.Sharpness(img).enhance(1.15)
        img = ImageEnhance.Brightness(img).enhance(self.config.BRIGHTNESS_BOOST)
        return img

    def _preprocess_traditional(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        scale = min(self.config.MAX_SIZE / max(w, h), 1.0)
        w, h = int(w * scale), int(h * scale)
        w = max(w, self.config.MIN_SIZE)
        h = max(h, self.config.MIN_SIZE)
        w, h = (w // 8) * 8, (h // 8) * 8
        img = img.resize((w, h), Image.LANCZOS)

        img = self._adaptive_gamma(img)
        img = self._boost_colors(img)
        return img

    def _hybrid_blend(self, traditional_img: Image.Image, diffusion_img: Image.Image,
                       blend_ratio: float) -> Image.Image:
        """Blends luminance channels in LAB space: blend_ratio weight on traditional output."""
        trad_array = np.array(traditional_img, dtype=np.float32)
        diff_array = np.array(diffusion_img, dtype=np.float32)

        trad_lab = cv2.cvtColor(trad_array.astype(np.uint8), cv2.COLOR_RGB2LAB).astype(np.float32)
        diff_lab = cv2.cvtColor(diff_array.astype(np.uint8), cv2.COLOR_RGB2LAB).astype(np.float32)

        blended_lab = trad_lab.copy()
        blended_lab[:, :, 0] = blend_ratio * trad_lab[:, :, 0] + (1 - blend_ratio) * diff_lab[:, :, 0]

        blended_rgb = cv2.cvtColor(blended_lab.astype(np.uint8), cv2.COLOR_LAB2RGB)
        return Image.fromarray(blended_rgb)

    # ------------------------------------------------------------------
    # CLIP-guided diffusion (Stage 2)
    # ------------------------------------------------------------------

    def _clip_guided_enhance(self, image: Image.Image, traditional_clip_emb: torch.Tensor,
                              seed: int = 42) -> Image.Image:
        """Runs DDIM img2img diffusion with periodic CLIP-gradient guidance toward the
        target prompt, constrained by similarity to the traditional-enhancement embedding."""
        generator = torch.Generator(device=self.device).manual_seed(seed)

        with torch.no_grad():
            img_tensor = self.pipe.image_processor.preprocess(image).to(
                device=self.device, dtype=self.dtype
            )
            latents = self.pipe.vae.encode(img_tensor).latent_dist.sample(generator)
            latents = latents * self.pipe.vae.config.scaling_factor

        with torch.no_grad():
            prompt_embeds = self.pipe._encode_prompt(
                self.config.TARGET_PROMPT,
                self.device,
                1,
                do_classifier_free_guidance=True,
                negative_prompt=self.config.NEGATIVE_PROMPT,
            )

        self.pipe.scheduler.set_timesteps(self.config.NUM_INFERENCE_STEPS, device=self.device)
        timesteps = self.pipe.scheduler.timesteps

        noise = torch.randn(latents.shape, generator=generator, device=self.device, dtype=self.dtype)
        strength = self.config.DIFFUSION_STRENGTH
        init_timestep = min(int(self.config.NUM_INFERENCE_STEPS * strength), self.config.NUM_INFERENCE_STEPS)
        t_start = max(self.config.NUM_INFERENCE_STEPS - init_timestep, 0)
        timesteps = timesteps[t_start:]
        latents = self.pipe.scheduler.add_noise(latents, noise, timesteps[0:1])

        for i, t in enumerate(timesteps):
            latent_model_input = torch.cat([latents] * 2)
            latent_model_input = self.pipe.scheduler.scale_model_input(latent_model_input, t)

            with torch.no_grad():
                noise_pred = self.pipe.unet(
                    latent_model_input, t, encoder_hidden_states=prompt_embeds
                ).sample

            noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
            noise_pred = noise_pred_uncond + self.config.GUIDANCE_SCALE * (noise_pred_text - noise_pred_uncond)

            # Periodic CLIP-gradient nudge toward the target prompt, weighted against
            # drift from the traditional-enhancement embedding.
            if i % 8 == 0 and i < len(timesteps) - 10:
                try:
                    latents_for_clip = latents.detach().requires_grad_(True)
                    current_img = self.pipe.vae.decode(
                        latents_for_clip / self.pipe.vae.config.scaling_factor
                    ).sample
                    current_img = (current_img / 2 + 0.5).clamp(0, 1)

                    clip_img = F.interpolate(current_img, size=(224, 224), mode="bilinear", align_corners=False)
                    clip_mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1).to(self.device)
                    clip_std = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1).to(self.device)
                    clip_img = (clip_img - clip_mean) / clip_std

                    current_clip_emb = self.clip_model.encode_image(clip_img)
                    current_clip_emb = current_clip_emb / current_clip_emb.norm(dim=-1, keepdim=True)

                    preservation = F.cosine_similarity(current_clip_emb, traditional_clip_emb, dim=1)
                    clip_sim = current_clip_emb @ self.pos_text.T
                    clip_loss = -clip_sim - self.config.PRESERVATION_WEIGHT * preservation

                    clip_grad = torch.autograd.grad(clip_loss.sum(), latents_for_clip, retain_graph=False)[0]
                    latents = latents - 1.5 * clip_grad.detach()
                except Exception:
                    # Guidance step is best-effort; skip on failure and continue denoising.
                    pass

            latents = self.pipe.scheduler.step(noise_pred, t, latents).prev_sample

        with torch.no_grad():
            image = self.pipe.vae.decode(latents / self.pipe.vae.config.scaling_factor).sample
            image = (image / 2 + 0.5).clamp(0, 1)
            image = self.pipe.image_processor.postprocess(image, output_type="pil")[0]

        return image

    # ------------------------------------------------------------------
    # Full pipeline
    # ------------------------------------------------------------------

    def enhance(self, image_path: str, output_path: Optional[str] = None):
        """Runs the full enhancement pipeline on a single image.

        Returns:
            (original_image, traditional_enhancement, final_output, iteration_logs)
        """
        raw_img = Image.open(image_path).convert("RGB")
        original_size = raw_img.size

        w, h = raw_img.size
        scale = min(self.config.MAX_SIZE / max(w, h), 1.0)
        w, h = int(w * scale), int(h * scale)
        w = max(w, self.config.MIN_SIZE)
        h = max(h, self.config.MIN_SIZE)
        w, h = (w // 8) * 8, (h // 8) * 8
        raw_resized = raw_img.resize((w, h), Image.LANCZOS)

        print("\nAnalyzing image brightness...")
        brightness_params = self._analyze_brightness(raw_resized)

        self.current_target_luminance = brightness_params["target_luminance"]
        self.current_gamma_max = brightness_params["gamma_max"]
        self.current_saturation_boost = brightness_params["saturation_boost"]
        self.current_contrast_boost = brightness_params["contrast_boost"]

        print(f"  Target luminance: {brightness_params['target_luminance']}")
        print(f"  Saturation boost: {brightness_params['saturation_boost']:.2f}x")
        print(f"  Contrast boost:   {brightness_params['contrast_boost']:.2f}x")
        print(f"  Brightness level: {brightness_params['brightness_level']}")

        # Stage 1: traditional enhancement
        print("\nStage 1: traditional enhancement (gamma + brightness)")
        trad_img = self._preprocess_traditional(raw_resized)
        traditional_clip = self._encode_image_clip(trad_img)

        # Stage 2: CLIP-guided diffusion refinement
        print(f"Stage 2: CLIP-guided diffusion ({self.config.NUM_ITERATIONS} iterations)")

        best_img = trad_img
        best_score = -1e9
        iteration_logs = []

        for iteration in range(self.config.NUM_ITERATIONS):
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            try:
                candidate = self._clip_guided_enhance(trad_img, traditional_clip, seed=100 + iteration)

                cand_clip = self._encode_image_clip(candidate)
                preservation = F.cosine_similarity(traditional_clip, cand_clip).item()
                clip_score, pos_sim, neg_sim = self._compute_clip_score(candidate)
                combined = clip_score + (0.5 * preservation)

                iteration_logs.append({
                    "iteration": iteration + 1,
                    "clip_score": round(clip_score, 4),
                    "pos_similarity": round(pos_sim, 4),
                    "neg_similarity": round(neg_sim, 4),
                    "preservation": round(preservation, 4),
                    "combined": round(combined, 4),
                })

                print(f"  Iter {iteration + 1}: clip={clip_score:.3f} "
                      f"preservation={preservation:.3f} combined={combined:.3f}")

                if preservation >= self.config.MIN_PRESERVATION_SCORE and combined > best_score:
                    best_score = combined
                    best_img = candidate
                    print(f"    -> new best (score={best_score:.3f})")
                elif preservation < self.config.MIN_PRESERVATION_SCORE:
                    print(f"    -> rejected (preservation {preservation:.3f} below threshold)")

            except Exception as e:
                iteration_logs.append({"iteration": iteration + 1, "error": str(e)})
                print(f"  Iter {iteration + 1}: error - {e}")
                continue

        # Stage 3: blend traditional and diffusion outputs
        print("\nStage 3: blending traditional output with diffusion refinement")
        final_img = self._hybrid_blend(trad_img, best_img, self.config.HYBRID_BLEND_RATIO)

        raw_out = raw_resized.resize(original_size, Image.LANCZOS)
        trad_out = trad_img.resize(original_size, Image.LANCZOS)
        final_out = final_img.resize(original_size, Image.LANCZOS)

        if output_path:
            final_out.save(output_path, quality=95)
            print(f"\nSaved enhanced image to: {output_path}")

        return raw_out, trad_out, final_out, iteration_logs


## 3. Run enhancement

Edit the configuration cell, then run the cell below.

In [ ]:
# Runtime configuration.
# Edit the values below to point at your own image/dataset and to adjust
# enhancement strength. No need to touch the pipeline code above.

class RunConfig(Config):
    # ---- Input/output ----
    TEST_MODE = True  # True: enhance a single image. False: batch-process a folder.
    TEST_IMAGE_PATH = "/content/drive/MyDrive/<your_project_folder>/sample_input.jpg"

    BATCH_INPUT_DIR = "/content/drive/MyDrive/<your_project_folder>/lol_dataset/eval15/low"
    BATCH_OUTPUT_DIR = "/content/drive/MyDrive/<your_project_folder>/enhanced"

    # ---- Brightness and diffusion strength ----
    BRIGHTNESS_BOOST = 1.3       # 1.0 = no change, 1.5 = 50% brighter
    DIFFUSION_STRENGTH = 0.30    # 0.0 = traditional only, 1.0 = maximum diffusion
    NUM_ITERATIONS = 4           # refinement passes

    # ---- Color and contrast ----
    SATURATION_BOOST = 1.30
    CONTRAST_BOOST = 1.3

    # ---- Semantic preservation ----
    MIN_PRESERVATION_SCORE = 0.85   # stricter = closer to traditional-enhancement output
    PRESERVATION_WEIGHT = 0.75

    # ---- Diffusion sampling ----
    GUIDANCE_SCALE = 3.0
    NUM_INFERENCE_STEPS = 50

    # ---- Gamma correction ----
    GAMMA_MIN = 0.6
    GAMMA_MAX = 1.2
    TARGET_LUMINANCE = 175

    # ---- Final blend ----
    HYBRID_BLEND_RATIO = 0.75   # fraction of luminance taken from traditional output
    DETAIL_PRESERVATION_STRENGTH = 0.85

    # ---- Brightness-adaptive thresholds ----
    VERY_DARK_THRESHOLD = 50
    DARK_THRESHOLD = 70
    MODERATE_DARK_THRESHOLD = 100


print("Configuration loaded.")
print("Edit TEST_IMAGE_PATH, BRIGHTNESS_BOOST, DIFFUSION_STRENGTH, "
      "SATURATION_BOOST, CONTRAST_BOOST, and NUM_ITERATIONS as needed.")


In [ ]:
# Run enhancement: single image (display inline) or batch (save to disk).

import gc
import os
import json
import datetime
import torch
from PIL import Image
import matplotlib.pyplot as plt

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading enhancement model (first run takes 1-2 minutes)...\n")
enhancer = CLIPGuidedDiffusionEnhancer(RunConfig)

if RunConfig.TEST_MODE:
    print("Mode: single image\n")

    if RunConfig.TEST_IMAGE_PATH is None or not os.path.exists(RunConfig.TEST_IMAGE_PATH):
        raise FileNotFoundError(
            f"Image not found: {RunConfig.TEST_IMAGE_PATH}\n"
            "Set TEST_IMAGE_PATH in the configuration cell above."
        )

    img_name = os.path.basename(RunConfig.TEST_IMAGE_PATH)
    print(f"Input: {RunConfig.TEST_IMAGE_PATH}")
    print("Settings:")
    print(f"  brightness_boost   = {RunConfig.BRIGHTNESS_BOOST}")
    print(f"  diffusion_strength = {RunConfig.DIFFUSION_STRENGTH}")
    print(f"  iterations         = {RunConfig.NUM_ITERATIONS}")
    print(f"  saturation_boost   = {RunConfig.SATURATION_BOOST}")
    print(f"  contrast_boost     = {RunConfig.CONTRAST_BOOST}\n")

    orig, traditional, final, logs = enhancer.enhance(
        RunConfig.TEST_IMAGE_PATH,
        output_path=None,
    )

    fig, axes = plt.subplots(1, 3, figsize=(20, 8))
    axes[0].imshow(orig)
    axes[0].set_title("Original input", fontsize=14, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(traditional)
    axes[1].set_title("Traditional enhancement\n(gamma + brightness)", fontsize=14, fontweight="bold")
    axes[1].axis("off")

    axes[2].imshow(final)
    axes[2].set_title("Final output\n(hybrid + diffusion refinement)", fontsize=14, fontweight="bold")
    axes[2].axis("off")

    plt.suptitle(f"Low-light enhancement: {img_name}", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print("Iteration metrics:")
    for log in logs:
        if "error" not in log:
            print(f"  [{log['iteration']}] clip={log['clip_score']} "
                  f"preservation={log['preservation']} combined={log['combined']}")
        else:
            print(f"  [{log['iteration']}] error: {log['error']}")

    print("\nDone. To save the output, pass output_path to enhancer.enhance().")

else:
    print("Mode: batch\n")

    input_dir = RunConfig.BATCH_INPUT_DIR
    output_dir = RunConfig.BATCH_OUTPUT_DIR
    os.makedirs(output_dir, exist_ok=True)

    print("Settings:")
    print(f"  brightness_boost   = {RunConfig.BRIGHTNESS_BOOST}")
    print(f"  diffusion_strength = {RunConfig.DIFFUSION_STRENGTH}")
    print(f"  iterations         = {RunConfig.NUM_ITERATIONS}\n")

    img_extensions = (".png", ".jpg", ".jpeg", ".bmp", ".tif")
    img_files = sorted(f for f in os.listdir(input_dir) if f.lower().endswith(img_extensions))

    if not img_files:
        raise FileNotFoundError(f"No images found in {input_dir}")

    print(f"Found {len(img_files)} images.")

    all_logs = []
    successful, failed = 0, 0

    for idx, img_file in enumerate(img_files):
        img_path = os.path.join(input_dir, img_file)
        out_path = os.path.join(output_dir, img_file)

        try:
            print(f"[{idx + 1}/{len(img_files)}] {img_file}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            _, _, _, logs = enhancer.enhance(img_path, out_path)
            all_logs.append({"filename": img_file, "status": "success", "iteration_logs": logs})
            successful += 1

        except Exception as e:
            failed += 1
            print(f"  error: {e}")
            all_logs.append({"filename": img_file, "status": "failed", "error": str(e)})

    print(f"\nBatch complete: {successful}/{len(img_files)} succeeded, {failed} failed "
          f"({100 * successful / len(img_files):.1f}% success rate)")
    print(f"Output directory: {output_dir}")

    logs_path = os.path.join(output_dir, "processing_logs.json")
    logs_data = {
        "timestamp": datetime.datetime.now().isoformat(),
        "total_images": len(img_files),
        "successful": successful,
        "failed": failed,
        "success_rate": round(100 * successful / len(img_files), 2),
        "settings": {
            "brightness_boost": RunConfig.BRIGHTNESS_BOOST,
            "diffusion_strength": RunConfig.DIFFUSION_STRENGTH,
            "iterations": RunConfig.NUM_ITERATIONS,
            "saturation_boost": RunConfig.SATURATION_BOOST,
            "contrast_boost": RunConfig.CONTRAST_BOOST,
        },
        "images": all_logs,
    }
    with open(logs_path, "w") as f:
        json.dump(logs_data, f, indent=2)
    print(f"Logs saved to: {logs_path}")


## 4. Evaluation

In [ ]:
# Evaluation against the LOL dataset (or any folder of paired low/high/enhanced images).
# Supports single-image and whole-folder evaluation, and computes PSNR, SSIM, LPIPS,
# MUSIQ, LOE, and FID where the relevant reference images are available.

import os
import cv2
import numpy as np
import torch
import pyiqa
from tqdm.notebook import tqdm
from pathlib import Path
import json
from datetime import datetime


class ImageQualityEvaluator:
    """Computes standard low-light enhancement metrics for single images or folders.

    Metric availability depends on which reference images are supplied:
      - MUSIQ:  no reference needed
      - PSNR/SSIM/LPIPS: needs a ground-truth (normal-light) image
      - LOE: needs the original low-light input
      - FID: computed once per folder pair (enhanced vs. ground truth), not per image
    """

    def __init__(self, device=None):
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Running on device: {self.device}")
        self.models = {}

    def _get_model(self, metric_name):
        """Lazily loads a pyiqa metric model, caching it for reuse."""
        if metric_name not in self.models:
            try:
                self.models[metric_name] = pyiqa.create_metric(metric_name, device=self.device)
            except Exception as e:
                print(f"Could not load metric '{metric_name}': {e}")
                return None
        return self.models[metric_name]

    def calculate_loe(self, input_path, enhanced_path, resize_dim=50):
        """Lightness Order Error: fraction of pairwise pixel brightness orderings
        that are inverted between the original and enhanced image. Lower is better."""
        img_input = cv2.imread(input_path)
        img_enhanced = cv2.imread(enhanced_path)

        if img_input is None or img_enhanced is None:
            print(f"Could not read images for LOE:\n  input:    {input_path}\n  enhanced: {enhanced_path}")
            return None

        L_in = np.max(img_input, axis=2)
        L_out = np.max(img_enhanced, axis=2)

        L_in_small = cv2.resize(L_in, (resize_dim, resize_dim)).flatten()
        L_out_small = cv2.resize(L_out, (resize_dim, resize_dim)).flatten()

        order_in = L_in_small[:, None] >= L_in_small[None, :]
        order_out = L_out_small[:, None] >= L_out_small[None, :]
        diff = np.logical_xor(order_in, order_out)

        return float(np.mean(np.sum(diff, axis=1)))

    def evaluate_single_image(self, enhanced_path, input_path=None, gt_path=None):
        """Evaluates one enhanced image against its (optional) input and ground truth."""
        results = {}

        musiq = self._get_model("musiq")
        if musiq:
            try:
                results["MUSIQ"] = round(musiq(enhanced_path).item(), 4)
            except Exception as e:
                print(f"MUSIQ failed: {e}")

        if gt_path and os.path.exists(gt_path):
            for metric in ["psnr", "ssim", "lpips"]:
                model = self._get_model(metric)
                if model:
                    try:
                        results[metric.upper()] = round(model(enhanced_path, gt_path).item(), 4)
                    except Exception as e:
                        print(f"{metric.upper()} failed: {e}")

        if input_path and os.path.exists(input_path):
            loe_val = self.calculate_loe(input_path, enhanced_path)
            if loe_val is not None:
                results["LOE"] = round(loe_val, 4)

        return results

    def evaluate_folder(self, enhanced_dir, low_dir=None, high_dir=None, output_json=None):
        """Evaluates every image in enhanced_dir against matching files (by filename)
        in low_dir and/or high_dir, then aggregates per-metric statistics and FID.
        """
        print(f"\nEvaluating folder: {enhanced_dir}")
        if low_dir:
            print(f"  low-light reference:  {low_dir}")
        if high_dir:
            print(f"  ground-truth reference: {high_dir}")

        enhanced_files = sorted(
            f for f in os.listdir(enhanced_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif"))
        )
        if not enhanced_files:
            print(f"No images found in {enhanced_dir}")
            return None

        print(f"Found {len(enhanced_files)} images.\n")

        agg_results = {"PSNR": [], "SSIM": [], "LPIPS": [], "MUSIQ": [], "LOE": []}
        individual_results = []

        for fname in tqdm(enhanced_files, desc="Evaluating"):
            enhanced_path = os.path.join(enhanced_dir, fname)
            low_path = os.path.join(low_dir, fname) if low_dir and os.path.exists(os.path.join(low_dir, fname)) else None
            high_path = os.path.join(high_dir, fname) if high_dir and os.path.exists(os.path.join(high_dir, fname)) else None

            metrics = self.evaluate_single_image(enhanced_path, low_path, high_path)
            individual_results.append({"filename": fname, "metrics": metrics})

            for metric, value in metrics.items():
                if metric in agg_results:
                    agg_results[metric].append(value)

        final_stats = {}
        print(f"\nAggregated results (n={len(enhanced_files)}):")
        for metric, values in agg_results.items():
            if values:
                final_stats[metric] = {
                    "average": round(float(np.mean(values)), 4),
                    "std": round(float(np.std(values)), 4),
                    "min": round(float(np.min(values)), 4),
                    "max": round(float(np.max(values)), 4),
                    "count": len(values),
                }
                print(f"  {metric}: avg={final_stats[metric]['average']} "
                      f"std={final_stats[metric]['std']} "
                      f"min={final_stats[metric]['min']} max={final_stats[metric]['max']}")

        if high_dir and os.path.exists(high_dir):
            print("\nComputing FID (dataset-level distribution similarity, lower is better)...")
            fid_model = self._get_model("fid")
            if fid_model:
                try:
                    fid_score = fid_model(enhanced_dir, high_dir).item()
                    final_stats["FID"] = {"score": round(fid_score, 4)}
                    print(f"  FID: {fid_score:.4f}")
                except Exception as e:
                    print(f"  FID failed: {e}")

        if output_json:
            output_data = {
                "timestamp": datetime.now().isoformat(),
                "enhanced_dir": enhanced_dir,
                "low_dir": low_dir,
                "high_dir": high_dir,
                "num_images": len(enhanced_files),
                "aggregated_stats": final_stats,
                "individual_results": individual_results,
            }
            with open(output_json, "w") as f:
                json.dump(output_data, f, indent=2)
            print(f"\nResults saved to: {output_json}")

        return final_stats

    def evaluate_lol_eval15(self, base_lol_path, enhanced_dir, output_json=None):
        """Convenience wrapper for the standard LOL/eval15/{low,high} layout."""
        low_dir = os.path.join(base_lol_path, "eval15", "low")
        high_dir = os.path.join(base_lol_path, "eval15", "high")
        if not os.path.exists(low_dir) or not os.path.exists(high_dir):
            print(f"LOL eval15 dataset not found at {base_lol_path}")
            return None
        return self.evaluate_folder(enhanced_dir, low_dir, high_dir, output_json)

    @staticmethod
    def print_metric_guide():
        guide = {
            "PSNR": ("full-reference", "higher is better", "pixel-level reconstruction accuracy"),
            "SSIM": ("full-reference", "higher is better", "structural/perceptual similarity"),
            "LPIPS": ("full-reference", "lower is better", "perceptual distance via deep features"),
            "MUSIQ": ("no-reference", "higher is better", "predicted perceptual quality"),
            "LOE": ("input-reference", "lower is better", "brightness-ordering preservation"),
            "FID": ("dataset-level", "lower is better", "distribution similarity to ground truth"),
        }
        print("Metric guide:")
        for metric, (kind, direction, desc) in guide.items():
            print(f"  {metric:6s} [{kind}, {direction}] - {desc}")


In [ ]:
# Run evaluation on a single image or a full folder.
# Edit the paths below, then run this cell.

import os

evaluator = ImageQualityEvaluator()
evaluator.print_metric_guide()

SINGLE_MODE = True  # False to evaluate a whole folder instead

# --- single-image mode ---
img_enhanced = "/content/drive/MyDrive/<your_project_folder>/enhanced_output.jpg"
img_input = "/content/drive/MyDrive/<your_project_folder>/sample_input.jpg"     # for LOE
img_gt = "/content/drive/MyDrive/<your_project_folder>/sample_ground_truth.jpg"  # for PSNR/SSIM/LPIPS

# --- folder mode ---
folder_enhanced = "/content/drive/MyDrive/<your_project_folder>/enhanced"
folder_low = "/content/drive/MyDrive/<your_project_folder>/lol_dataset/eval15/low"
folder_high = "/content/drive/MyDrive/<your_project_folder>/lol_dataset/eval15/high"
output_json = "/content/drive/MyDrive/<your_project_folder>/evaluation_results.json"

if SINGLE_MODE:
    if not os.path.exists(img_enhanced):
        print(f"File not found: {img_enhanced}")
        print("Update img_enhanced (and img_input/img_gt if available) above.")
    else:
        scores = evaluator.evaluate_single_image(img_enhanced, img_input, img_gt)
        print("\nSingle-image results:")
        for metric, value in scores.items():
            print(f"  {metric:6s}: {value:.4f}")

else:
    if not os.path.exists(folder_enhanced):
        print(f"Folder not found: {folder_enhanced}")
    else:
        stats = evaluator.evaluate_folder(folder_enhanced, folder_low, folder_high, output_json)
